# XMCA-Net-Lite: 08 - Comprehensive Ablation Study
### Purpose
Execute the full required ablation suite across 5 seeds:
1. Full model ($S_P + S_R + S_U$)
2. Remove $S_P$
3. Remove $S_R$
4. Remove $S_U$
5. Replace cross-modal utilization with text-only NLI
6. Replace learned isotonic retrieval fusion with fixed weighted fusion
7. Replace learned calibrated combiner with heuristic baseline: $1 - \min(S_P, S_R, S_U)$
8. Cross-VLM transfer: Train on Qwen2-VL, test directly on LLaVA-OneVision.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.evaluation.ablation_suite import run_ablation_experiment
from src.utils.checkpoints import ShardManager
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seeds = config["project"]["random_seeds"]
set_seed(config["project"]["default_seed"])


In [ ]:
# Load unified dataset records
p_df = ShardManager(Path(config["paths"]["perception_dir"])).load_all_shards("perception_scores")
r_df = ShardManager(Path(config["paths"]["retrieval_dir"])).load_all_shards("retrieval_scores")
u_df = ShardManager(Path(config["paths"]["utilization_dir"])).load_all_shards("utilization_scores")

unified = p_df[["id", "dataset", "s_p", "is_hallucination_gt"]].merge(
    r_df[["id", "s_r", "s_r_fixed", "retrieved_passages"]], on="id"
).merge(
    u_df[["id", "s_u", "s_u_text_only"]], on="id"
)
unified["evidence_count"] = unified["retrieved_passages"].apply(len)
unified["claim_count"] = 2

splits_dir = Path(config["paths"]["splits_dir"])
train_ids = set()
val_ids = set()
test_ids = set()

for p in splits_dir.glob("*/*train.json"):
    with open(p) as f:
        train_ids.update([r["id"] for r in json.load(f)])
for p in splits_dir.glob("*/*val.json"):
    with open(p) as f:
        val_ids.update([r["id"] for r in json.load(f)])
for p in splits_dir.glob("*/*test.json"):
    with open(p) as f:
        test_ids.update([r["id"] for r in json.load(f)])

train_rec = unified[unified["id"].isin(train_ids)].to_dict(orient="records")
val_rec = unified[unified["id"].isin(val_ids)].to_dict(orient="records")
test_rec = unified[unified["id"].isin(test_ids)].to_dict(orient="records")


In [ ]:
# Run all ablation conditions
ablation_names = [
    "full_model",
    "no_perception",
    "no_retrieval",
    "no_utilization",
    "text_only_utilization",
    "fixed_weighted_fusion",
    "min_rule_baseline",
    "cross_vlm_transfer"
]

all_ablation_results = []

for name in ablation_names:
    print(f"
Running ablation experiment: {name}...")
    
    # Deep copy records to prevent mutating shared training records across iterations
    cur_train = [dict(r) for r in train_rec]
    cur_val = [dict(r) for r in val_rec]
    cur_test = [dict(r) for r in test_rec]
    
    if name == "text_only_utilization":
        for r in cur_train + cur_val + cur_test:
            r["s_u"] = r.get("s_u_text_only", r.get("s_u", 0.5))
    elif name == "fixed_weighted_fusion":
        for r in cur_train + cur_val + cur_test:
            r["s_r"] = r.get("s_r_fixed", r.get("s_r", 0.5))
    elif name == "cross_vlm_transfer":
        llava_gen_dir = Path(config["paths"]["generations_dir"]) / "llava_transfer"
        llava_df = ShardManager(llava_gen_dir).load_all_shards("llava_generation")
        if not llava_df.empty:
            llava_ids = set(llava_df["id"])
            llava_test = [r for r in cur_test if r["id"] in llava_ids]
            if llava_test:
                cur_test = llava_test
            
    res = run_ablation_experiment(
        train_records=cur_train,
        val_records=cur_val,
        test_records=cur_test,
        ablation_name=name,
        seeds=seeds
    )
    all_ablation_results.append(res)
    print(f"  AUROC: {res['auroc_mean']:.4f} +/- {res['auroc_std']:.4f} | ECE: {res['ece_mean']:.4f}")

out_dir = Path(config["paths"]["ablations_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "ablation_results.json", "w", encoding="utf-8") as f:
    json.dump(all_ablation_results, f, indent=2)

pd.DataFrame(all_ablation_results).to_parquet(out_dir / "ablation_table.parquet", index=False)
save_run_metadata(out_dir, config, config["project"]["default_seed"], "08_ablation", {"ablations": ablation_names})
print("
Step 08 completed. All ablation experiments saved.")
